# 07 · Generación del conjunto de test propio de keypoints

**Objetivo:** seleccionar 50 frames nuevos de Bundesliga (distintos a los 120 del fine-tuning), auto-etiquetarlos con el modelo C (el mejor del experimento de congelación) y prepararlos para revisión en CVAT, con el fin de construir un conjunto de test independiente.

**Contexto:** los modelos de keypoints se evaluaron hasta ahora sobre validación, que forma parte de los datos de entrenamiento. Para una evaluación rigurosa se construye un conjunto de test propio que ningún modelo ha visto. Se usa el modelo C para el auto-etiquetado porque predice mejor sobre Bundesliga que el modelo base, reduciendo el trabajo de corrección.

**Entrada:** modelo `keypoints_ft_C.pt` y frames en `data/frames/`.

**Salida:** 50 frames auto-etiquetados en `data/keypoints_test_autolabel/`, listos para CVAT.

In [1]:
from ultralytics import YOLO
from pathlib import Path
import random
import shutil

REPO_ROOT = Path("../..").resolve()
MODEL_C = REPO_ROOT / "models" / "keypoints_ft_C.pt"
FRAMES_DIR = REPO_ROOT / "data" / "frames"
OUTPUT_DIR = REPO_ROOT / "data" / "keypoints_test_autolabel"

N_TEST = 50

# Reconstruir el set de frames ya usados (los 120 del fine-tuning)
usados = set()
for split in ["train", "valid"]:
    for img in (REPO_ROOT / "data" / "bundesliga_keypoints" / split / "images").glob("*.jpg"):
        usados.add(img.name)

print(f"Frames ya usados: {len(usados)}")
print(f"Modelo C existe: {MODEL_C.exists()}")

Frames ya usados: 120
Modelo C existe: True


## 1. Selección de 50 frames nuevos

Se recorren todos los frames disponibles y se construye el nombre que tendrían en el formato del dataset (con prefijo de carpeta). Se descartan los ya usados en el fine-tuning y se seleccionan 50 nuevos, repartidos entre los distintos vídeos para asegurar variedad.

In [2]:
# Construir lista de frames candidatos (no usados)
candidatos = []
subdirs = [d for d in FRAMES_DIR.iterdir() if d.is_dir()]

for d in subdirs:
    for frame in d.glob("*.jpg"):
        # Nombre en formato dataset: carpeta_carpeta_archivo (como en el fine-tuning)
        nombre_dataset = f"{d.name}_{d.name}_{frame.name}".replace(" ", "_")
        if nombre_dataset not in usados:
            candidatos.append((frame, nombre_dataset))

print(f"Frames candidatos (no usados): {len(candidatos)}")

# Seleccionar 50 repartidos por vídeo
random.seed(123)  # semilla distinta a la del fine-tuning
random.shuffle(candidatos)
seleccionados = candidatos[:N_TEST]

print(f"Frames seleccionados para test: {len(seleccionados)}")

Frames candidatos (no usados): 756
Frames seleccionados para test: 50


In [3]:
# Auto-etiquetar con el modelo C
model = YOLO(str(MODEL_C))

images_out = OUTPUT_DIR / "images"
labels_out = OUTPUT_DIR / "labels"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

CONF_MIN = 0.5
N_KP = 24
procesados = 0

for frame_path, nombre_dataset in seleccionados:
    res = model(str(frame_path), verbose=False, conf=0.25)
    r = res[0]

    if r.keypoints is None or len(r.boxes) == 0:
        continue

    kps_xy = r.keypoints.xyn[0].cpu().numpy()
    kps_conf = r.keypoints.conf[0].cpu().numpy()

    partes = ["0", "0.5", "0.5", "1.0", "1.0"]
    for (x, y), c in zip(kps_xy, kps_conf):
        if c > CONF_MIN:
            partes.extend([f"{x:.6f}", f"{y:.6f}", "2"])
        else:
            partes.extend(["0", "0", "0"])

    shutil.copy2(frame_path, images_out / nombre_dataset)
    label_name = nombre_dataset.rsplit(".", 1)[0] + ".txt"
    with open(labels_out / label_name, "w") as f:
        f.write(" ".join(partes) + "\n")

    procesados += 1

print(f"Frames auto-etiquetados: {procesados}")
print(f"Guardados en: {OUTPUT_DIR}")

Frames auto-etiquetados: 50
Guardados en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\keypoints_test_autolabel


## 2. Conversión a COCO para CVAT

Se reutiliza la misma lógica del notebook 04 para convertir estas predicciones al formato COCO Keypoints e importarlas en CVAT.

In [4]:
import json
import cv2

CVAT_OUT = REPO_ROOT / "data" / "cvat_import_test"
CVAT_OUT.mkdir(parents=True, exist_ok=True)

KEYPOINT_NAMES = [
    "corner_inf_izq", "corner_sup_izq", "corner_inf_der", "corner_sup_der",
    "area_grande_izq_inf", "area_grande_izq_sup", "area_grande_der_inf", "area_grande_der_sup",
    "area_peq_izq_inf", "area_peq_izq_sup", "area_peq_der_inf", "area_peq_der_sup",
    "penalti_izq", "penalti_der", "centro", "media_inf", "media_sup",
    "fondo_izq_area_inf", "fondo_izq_area_sup", "fondo_der_area_inf",
    "poste_izq_inf", "poste_izq_sup", "poste_der_inf", "poste_der_sup",
]

coco = {
    "info": {"description": "Test keypoints Bundesliga"},
    "licenses": [], "images": [], "annotations": [],
    "categories": [{"id": 1, "name": "field", "supercategory": "field",
                    "keypoints": KEYPOINT_NAMES, "skeleton": []}]
}

ann_id = 1
img_id = 1
for label_path in sorted(labels_out.glob("*.txt")):
    img_path = images_out / (label_path.stem + ".jpg")
    if not img_path.exists():
        continue
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    coco["images"].append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})

    with open(label_path) as f:
        vals = f.read().split()
    kp_vals = vals[5:]
    keypoints_coco = []
    num_vis = 0
    for i in range(N_KP):
        x_n, y_n, v = float(kp_vals[i*3]), float(kp_vals[i*3+1]), int(kp_vals[i*3+2])
        if v == 2:
            keypoints_coco.extend([x_n*w, y_n*h, 2]); num_vis += 1
        else:
            keypoints_coco.extend([0, 0, 0])
    coco["annotations"].append({
        "id": ann_id, "image_id": img_id, "category_id": 1,
        "keypoints": keypoints_coco, "num_keypoints": num_vis,
        "bbox": [0,0,w,h], "area": w*h, "iscrowd": 0})
    ann_id += 1; img_id += 1

with open(CVAT_OUT / "annotations_coco.json", "w") as f:
    json.dump(coco, f, indent=2)

cvat_imgs = CVAT_OUT / "images"
cvat_imgs.mkdir(exist_ok=True)
for img_path in images_out.glob("*.jpg"):
    shutil.copy2(img_path, cvat_imgs / img_path.name)

print(f"Imágenes: {len(coco['images'])}, Anotaciones: {len(coco['annotations'])}")
print(f"Listo para CVAT en: {CVAT_OUT}")

Imágenes: 50, Anotaciones: 50
Listo para CVAT en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\cvat_import_test
